# Module 14: Hands-On Walkthrough
## CS 5045: Computation for the Data Sciences
**Looking Ahead: Machine Learning: Self-Guided Notebook**

This week has no assignment and no live Wednesday session (Wednesday 12/2 is Seminar 5
rehearsal), so this notebook is where you work with the ideas from the video. It is not graded.
It follows the same rhythm as a Wednesday session: try something first, work through it, then
connect it back to the lecture. Because no one is available to answer questions in real time, it
explains each step in more detail than a live demo would, and it includes five **Your Turn**
exercises with answers you can reveal after you try them.

For each step, the computational operation is named *before* the code.

**Part 1 (about 60 minutes): a new problem**

| Step | Operation | What We Do |
|------|-----------|------------|
| 1 | Represent | Try it first: a rule you already know, learned from examples |
| 2 | Represent | A messier problem: a building's daily energy use |
| 3 | Split | Hold out a test set before fitting anything |
| 4 | Transform (learned) | `model.fit()` finds the rule |
| 5 | Aggregate | R-squared and MAE summarize the test-set misses |
| 6 | Aggregate | Compare against a baseline: is the model better than guessing? |
| 7 | Transform (learned) | Leave out an input and see what changes |
| 8 | Filter | Find the biggest misses, and ask why |
| 9 | Communicate | Plot predicted vs. actual |

**Part 2 (optional, about 20 minutes):** run the exact model from the lecture video on the
Module 9 air-quality data, and check your numbers against the slides.

**At the end:** Check Your Understanding questions and a table of common errors and their fixes.

**Recap in one sentence:** all semester you wrote the rule; this week the computer finds the rule
from examples, and you judge it on data it has never seen.

**Before you start.** This notebook uses scikit-learn, a library you have not needed before. If the
first code cell fails with `ModuleNotFoundError: No module named 'sklearn'`, run
`%pip install scikit-learn` in a new cell, then restart the kernel. Note that the package is
*installed* as `scikit-learn` but *imported* as `sklearn`.

**Data:** Part 1 defines its data inside the notebook. Part 2 reads `data.csv` (the Module 9
air-quality file), which sits in the same `M14/` folder.

---
## Try It First: Can You Recover a Rule You Already Know?

Back on slide 2, "convert Celsius to Fahrenheit" was a rule *you* wrote. Below are five
temperature readings in both units, with no formula attached.

> **Pause:** Before running anything, look at the five pairs and write down the rule that
> connects them. What do you multiply by? What do you add?

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression

# REPRESENT: five examples where the right answer is already known
temps = pd.DataFrame({
    "celsius":    [0, 10, 20, 30, 37],
    "fahrenheit": [32.0, 50.0, 68.0, 86.0, 98.6],
})
temps

In [ ]:
# TRANSFORM (learned): hand the model the examples, not the formula
rule = LinearRegression()
rule.fit(temps[["celsius"]], temps["fahrenheit"])

print(f"Learned multiplier: {rule.coef_[0]:.2f}")
print(f"Learned intercept:  {rule.intercept_:.2f}")

# Apply the learned rule to a temperature it never saw
boiling = pd.DataFrame({"celsius": [100]})
print(f"Prediction for 100 C: {rule.predict(boiling)[0]:.1f} F")

**Discuss:** the model found 1.8 and 32 on its own, the same rule you would have written by hand.
Nobody told it the formula; it worked backward from five examples.

### What `fit()` and `predict()` just did

Two lines carry the whole idea of this module, so it is worth slowing down on them.

- **`rule.fit(inputs, answers)`** is given examples where the answer is already known. It searches
  for the multiplier and intercept that make the predictions as close as possible to those known
  answers. Technically, it makes the total of the squared misses as small as possible. The result
  is stored inside `rule`: `rule.coef_` holds the multiplier and `rule.intercept_` holds the
  intercept.
- **`rule.predict(new_inputs)`** does no searching at all. It plugs new inputs into the rule that
  `fit()` already found: here, 1.8 times the Celsius value, plus 32.

In short, `fit()` is where the learning happens, and `predict()` only applies what was learned.

**Why the double brackets?** Look at `temps[["celsius"]]` in the code. Single brackets,
`temps["celsius"]`, return one column as a list-like Series. Double brackets return a *table* with
one column. scikit-learn always expects the inputs as a table, because most models use several
input columns at once. The answers (`temps["fahrenheit"]`) use single brackets, because there is
only ever one target. Forgetting the second pair of brackets is the most common error in this
notebook; see the Common Errors table at the end.

---
## Your Turn 1: Recover a Second Rule

The table below lists five distances in both miles and kilometers.

1. Fit a `LinearRegression` that predicts `km` from `miles`, using the same pattern as the Celsius
   example.
2. Print the learned multiplier and intercept. What rule did it find?
3. Use the fitted model to convert a half marathon, 13.1 miles, to kilometers.

In [ ]:
distances = pd.DataFrame({
    "miles": [1, 5, 10, 26.2, 50],
    "km":    [1.609, 8.047, 16.093, 42.165, 80.467],
})

# Your code here: fit, print the weights, then predict 13.1 miles

<details><summary>Show the answer</summary>

```python
converter = LinearRegression()
converter.fit(distances[["miles"]], distances["km"])

print(f"Learned multiplier: {converter.coef_[0]:.3f}")
print(f"Learned intercept:  {converter.intercept_:.3f}")

half_marathon = pd.DataFrame({"miles": [13.1]})
print(f"13.1 miles = {converter.predict(half_marathon)[0]:.2f} km")
```

The multiplier is about 1.609 and the intercept is essentially 0, so the rule is
"kilometers = 1.609 times miles." A half marathon comes out to about 21.08 km. Unlike the
temperature rule, this one has no added constant, and the model discovered that too: zero miles is
zero kilometers.

</details>

Real data is never this clean. Both conversions are exact rules, so the examples fit them
perfectly. The next problem is the realistic case: the rule is only approximate, and some days
have causes the model will never see.

---
## Represent: A Building's Daily Energy Use

A facilities analyst wants to predict a building's daily electricity use (`kwh`) from two
numbers recorded every day: the outdoor temperature (`temp_c`) and how many people badged in
(`occupancy`).

The cell below simulates 60 days of readings. It also writes a `note` column: the maintenance
log. **The model will never be given the `note` column.** We keep it so that, at the end, we can
check whether the model's worst mistakes line up with something it was never told.

In [ ]:
import numpy as np

# REPRESENT: simulate 60 days of readings (fixed seed, so everyone gets the same numbers)
rng = np.random.default_rng(5045)
n_days = 60

temp_c    = np.round(rng.uniform(12, 34, n_days), 1)
occupancy = rng.integers(40, 220, n_days)
kwh       = np.round(150 + 5.0 * temp_c + 0.9 * occupancy + rng.normal(0, 12, n_days), 1)

energy = pd.DataFrame({
    "day": range(1, n_days + 1),
    "temp_c": temp_c,
    "occupancy": occupancy,
    "kwh": kwh,
    "note": "",
})

# The maintenance log: three days where something the sensors don't capture changed the usage
log = [
    (14,  70, "HVAC fault: chiller stuck on"),
    (21,  60, "HVAC fault: chiller stuck on"),
    (47, -55, "Scheduling error: HVAC ran weekend hours"),
]
for day, change, note in log:
    energy.loc[energy["day"] == day, "kwh"] += change
    energy.loc[energy["day"] == day, "note"] = note

print(f"{len(energy)} days, columns: {list(energy.columns)}")
energy.head()

---
## Split: Hold Out a Test Set Before Fitting Anything

Same move as the lecture: the model learns from the training rows and is graded on rows it never
touched. We hold out 25% of the days.

> **Pause:** Why not fit on all 60 days and then score the model on those same 60 days?
> What would a high score on the training rows actually prove?

In [ ]:
from sklearn.model_selection import train_test_split

# The inputs (X) and the target (y). Note: no 'note' column in X.
X = energy[["temp_c", "occupancy"]]
y = energy["kwh"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42)

print(f"Training days: {len(X_train)}")
print(f"Test days:     {len(X_test)}")

**What the split produced.** `train_test_split` returns four pieces, always in this order:

| Name | Contents | Used for |
|------|----------|----------|
| `X_train` | inputs for 45 days | fitting |
| `X_test` | inputs for 15 days | predicting, then scoring |
| `y_train` | known `kwh` for the same 45 days | fitting |
| `y_test` | known `kwh` for the same 15 days | scoring only |

Two settings deserve a note. `test_size=0.25` holds out a quarter of the rows. `random_state=42`
fixes the shuffle, so everyone who runs this notebook gets the same 15 test days and the same
numbers. Change it and you will get a different split and slightly different results, which is a
useful reminder that a score computed on 15 days carries some luck.

The rule that matters most: **the test rows must not influence the fit in any way.** If they do,
the test score stops being an honest measure of how the model handles new data.

---
## Transform (learned): `model.fit()` Finds the Rule

One model, two inputs, chosen in advance. `model.fit()` searches for the weights on `temp_c`
and `occupancy` that come closest to the real `kwh` values in the training set.

> **Pause:** Before running, guess: will more people in the building push the predicted usage
> up or down? Will hotter days?

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print(f"kWh per degree C:   {model.coef_[0]:.2f}")
print(f"kWh per person:     {model.coef_[1]:.2f}")
print(f"Baseline (intercept): {model.intercept_:.1f} kWh")

**Discuss:** read the weights as a sentence. Each extra degree adds about 5 kWh, and each extra
person adds a bit under 1 kWh. Nobody wrote that rule; the model found it from 45 days of
examples.

---
## Your Turn 2: Use the Model

1. Use the trained `model` to predict energy use for a day you make up: 30 degrees C with 150
   people in the building. Build a one-row DataFrame with the columns `temp_c` and `occupancy`, in
   that order, then call `model.predict()` on it.
2. Check your prediction by hand using the weights printed above:
   intercept + (weight per degree times 30) + (weight per person times 150).

In [ ]:
# Your prediction for a 30 C day with 150 people:

<details><summary>Show the answer</summary>

```python
new_day = pd.DataFrame({"temp_c": [30], "occupancy": [150]})
print(f"Predicted use: {model.predict(new_day)[0]:.1f} kWh")
```

The model predicts about **438.0 kWh**. By hand: 154.8 + (4.90 times 30) + (0.91 times 150) is
roughly 154.8 + 147.0 + 136.5 = 438.3. The small difference comes from rounding the printed weights.
This is the whole of `predict()`: it is the same arithmetic you just did, applied to every row.

If you got a `ValueError` about feature names, check that the column names are spelled exactly
`temp_c` and `occupancy` and appear in that order. The model remembers the names it was fit on.

</details>

---
## Aggregate: How Good Is It on Days It Never Saw?

`model.predict()` applies the learned rule to the 15 held-out days. Then two numbers collapse
all 15 misses into one summary each, exactly as in the lecture.

> **Pause:** Predict before you run: will the training score or the test score be higher? Why?

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error

predictions = model.predict(X_test)

print(f"R-squared on training days: {r2_score(y_train, model.predict(X_train)):.3f}")
print(f"R-squared on test days:     {r2_score(y_test, predictions):.3f}")
print(f"MAE on test days:           {mean_absolute_error(y_test, predictions):.1f} kWh")

**Discuss:** the test score is lower than the training score. That is normal, and it is the
reason we split: the training score flatters the model, and the test score is the honest one.
An MAE of about 17.5 kWh means a typical prediction is off by roughly 17.5 kWh on a day the model
has never seen.

---
## Aggregate: Is the Model Better Than Guessing?

An R-squared of 0.811 sounds good, but good compared with what? Every model should be compared with
the simplest possible approach. Here that is to ignore temperature and occupancy entirely and predict
the **average training-day usage** for every test day. This is called a *baseline*.

> **Pause:** Will the baseline's MAE be a little worse than the model's 17.5 kWh, or a lot worse?

In [ ]:
# The baseline: predict the training-day average for every test day
average_use = y_train.mean()
baseline_predictions = [average_use] * len(y_test)

print(f"Average training-day use: {average_use:.1f} kWh")
print()
print(f"Baseline  R-squared: {r2_score(y_test, baseline_predictions):6.3f}   MAE: {mean_absolute_error(y_test, baseline_predictions):5.1f} kWh")
print(f"Model     R-squared: {r2_score(y_test, predictions):6.3f}   MAE: {mean_absolute_error(y_test, predictions):5.1f} kWh")

**Discuss:** guessing the average misses by about 51 kWh on a typical test day; the model misses by
about 17.5. Temperature and occupancy cut the typical error by roughly two-thirds.

The baseline also explains what R-squared measures. **R-squared compares a model with exactly this
baseline.** A score of 0 means "no better than predicting the average." A score of 1 means perfect
predictions. The baseline itself scores about 0 (here slightly below 0, because the training average
is not exactly the test-day average). So the model's 0.811 can be read as: the model removes about
81% of the error that guessing the average would leave.

A negative R-squared is possible, and it is a warning sign: the model is doing *worse* than
predicting the average.

---
## Transform (learned): What If We Leave Out an Input?

The model was given two inputs. Did it need both? One way to find out is to fit a second model with
temperature alone and compare the scores on the same 15 test days.

> **Pause:** Temperature had the larger weight (about 4.9 kWh per degree, against 0.9 per person).
> Will the temperature-only model do almost as well as the full model?

In [ ]:
temp_only = LinearRegression()
temp_only.fit(X_train[["temp_c"]], y_train)
temp_only_predictions = temp_only.predict(X_test[["temp_c"]])

print(f"Temperature only  R-squared: {r2_score(y_test, temp_only_predictions):.3f}   MAE: {mean_absolute_error(y_test, temp_only_predictions):.1f} kWh")
print(f"Both inputs       R-squared: {r2_score(y_test, predictions):.3f}   MAE: {mean_absolute_error(y_test, predictions):.1f} kWh")

**Discuss:** without occupancy, the model falls apart: R-squared drops from 0.811 to 0.232, and the
typical miss grows from about 17.5 to about 44 kWh.

That may be surprising, since temperature had the larger weight. **A larger weight does not mean a
more important input.** A weight is measured per unit, and the units differ. Temperature varies over
about 22 degrees in this data, so it moves usage by about 4.9 times 22, roughly 110 kWh. Occupancy
varies over about 180 people, so it moves usage by about 0.9 times 180, roughly 160 kWh. Occupancy
accounts for more of the day-to-day differences, even though its weight is smaller. (Part 2 has the
same issue with the `ozone` weight.)

---
## Your Turn 3: Leave Out the Other Input

Fit a model that uses **occupancy only**, and compare its test R-squared and MAE with the full model.
Before you run it, predict: will it be closer to the temperature-only model or to the full model?

In [ ]:
# Your occupancy-only model here:

<details><summary>Show the answer</summary>

```python
occupancy_only = LinearRegression()
occupancy_only.fit(X_train[["occupancy"]], y_train)
occupancy_only_predictions = occupancy_only.predict(X_test[["occupancy"]])

print(f"R-squared: {r2_score(y_test, occupancy_only_predictions):.3f}")
print(f"MAE:       {mean_absolute_error(y_test, occupancy_only_predictions):.1f} kWh")
```

R-squared is about **0.856** and MAE about **17.7 kWh**. The occupancy-only model is close to the full
model, and its R-squared is even slightly *higher*.

That does not mean temperature is useless. The data was simulated with a real temperature effect of
5 kWh per degree, and the full model recovered it (4.9). Two things are going on:

- **The test set is small.** With only 15 days, a few unusual days can move the score noticeably.
  The two models' MAEs are nearly identical (17.5 against 17.7), so the R-squared difference is
  within the range that luck in the split could produce.
- **Two of the 15 test days are maintenance-log days**, with errors far larger than anything
  temperature could explain. Those days dominate both scores.

The lesson: do not choose between models based on a small difference in one score on a small test
set. Large differences, such as 0.23 against 0.81, are meaningful. Small ones usually are not. Try
changing `random_state` in the split cell and rerunning to see how much the scores move.

</details>

---
## Filter: Find the Biggest Misses, and Ask Why

Now we line each test-day prediction up with what actually happened, keep the three largest
misses, and, for the first time, bring back the `note` column the model never saw.

> **Pause:** Before running, guess which days will be the worst misses.

In [ ]:
# Line up actual vs. predicted for each test day, then attach the maintenance log
results = energy.loc[X_test.index, ["day", "temp_c", "occupancy", "kwh", "note"]].copy()
results["predicted"] = predictions.round(1)
results["miss"] = (results["predicted"] - results["kwh"]).round(1)

# FILTER: keep the three largest misses, by size, ignoring direction
worst = results.reindex(results["miss"].abs().sort_values(ascending=False).index).head(3)
worst

**Discuss:** the two worst misses are exactly the two test days in the maintenance log.

- **Day 14:** the chiller was stuck on. The model predicted about 377 kWh; the building used 438.
- **Day 47:** the HVAC ran weekend hours by mistake. The model predicted about 492; the building used 442.

The model only ever saw temperature and occupancy. Whatever actually drove those days (a stuck
chiller, a scheduling mistake) was invisible to it. A model's mistakes often point at what it was
never given.

Notice the third miss: day 6 has no note at all. Not every miss has a story; some of it is
ordinary day-to-day noise. The useful habit is to *check* the big misses, not to assume each one
has a hidden cause.

---
## Your Turn 4: Filter the Misses by Direction

In `results`, `miss` is computed as **predicted minus actual**. A positive miss means the model
predicted too high; a negative miss means it predicted too low.

1. Keep only the test days where the model predicted **too low by more than 10 kWh**. This is the
   same Filter operation from Module 3, applied to a column the model produced.
2. If you were the facilities analyst, what is **one** more thing you would record every day, based
   on the misses you have seen?

In [ ]:
# Your filter here:

<details><summary>Show the answer</summary>

```python
too_low = results[results["miss"] < -10]
too_low
```

Three days qualify: **day 13** (about 14 kWh too low), **day 14** (about 61 kWh too low, the stuck
chiller), and **day 32** (about 13 kWh too low). Only one of the three has a maintenance-log entry;
the other two are ordinary noise of about the size MAE would lead you to expect.

For the second question, any daily record of what the HVAC system was actually doing would help:
equipment run hours, whether a fault alarm fired, or whether the day was scheduled as a weekday or a
weekend. The two worst misses were both HVAC problems that temperature and occupancy could not reveal.

</details>

---
## Communicate: Predicted vs. Actual

Points on the dashed line are perfect predictions. Points far from it are the misses we just
read about.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(results["kwh"], results["predicted"], color="#861F41")

# Dashed line: where a perfect prediction would land
both = pd.concat([results["kwh"], results["predicted"]])
lo, hi = both.min() - 10, both.max() + 10
ax.plot([lo, hi], [lo, hi], linestyle="--", color="gray")

# Label the days that have a maintenance-log entry
for _, row in results[results["note"] != ""].iterrows():
    ax.annotate(f"Day {row['day']}", (row["kwh"], row["predicted"]),
                textcoords="offset points", xytext=(8, -4))

ax.set_xlabel("Actual kWh")
ax.set_ylabel("Predicted kWh")
ax.set_title("Test days: predicted vs. actual energy use")
plt.tight_layout()
plt.show()

---
## Part 2 (optional): Run the Lecture's Model Yourself

Everything you saw in the video, now in your own hands: the same 300 days of air-quality
readings from Module 9, the same three inputs (`ozone`, `temp_c`, `humidity`), the same target
(`pm25`), and the same split. Your numbers should match slides 10 and 12 exactly.

In [ ]:
from pathlib import Path

# REPRESENT: the Module 9 air-quality data (same folder on GitHub; ../Datasets locally)
data_path = Path("data.csv") if Path("data.csv").exists() else Path("../Datasets/data.csv")
air = pd.read_csv(data_path)

print(f"{len(air)} rows, columns: {list(air.columns)}")
air.head()

In [ ]:
# SPLIT: hold out 20% of the days, same settings as the lecture
X_air = air[["ozone", "temp_c", "humidity"]]
y_air = air["pm25"]

Xa_train, Xa_test, ya_train, ya_test = train_test_split(
    X_air, y_air, test_size=0.2, random_state=42)

print(f"Training days: {len(Xa_train)}")
print(f"Test days:     {len(Xa_test)}")

In [ ]:
# TRANSFORM (learned): one pre-chosen model, fit on the training days only
air_model = LinearRegression()
air_model.fit(Xa_train, ya_train)

for name, weight in zip(X_air.columns, air_model.coef_):
    print(f"Weight on {name:<9} {weight:8.3f}")
print(f"Intercept:          {air_model.intercept_:8.3f}")

**Discuss:** the weight on `ozone` looks enormous next to the other two. That is about units, not
importance: ozone is measured in small numbers (around 0.05), so a rise of 0.01 adds only about
1.2 to the predicted pm25. Warmer days nudge the prediction down slightly, and more humid days
nudge it up. Nobody wrote these numbers; `fit()` found them from 240 days of examples.

In [ ]:
# AGGREGATE: score the model on the 60 days it never saw
air_predictions = air_model.predict(Xa_test)

print(f"R-squared: {r2_score(ya_test, air_predictions):.3f}")
print(f"MAE:       {mean_absolute_error(ya_test, air_predictions):.2f} micrograms per cubic meter")

**Check against slide 10:** R-squared 0.330 and MAE 2.38. A moderate fit, not an impressive one,
and still an honest, useful result: three environmental variables explain only part of air
quality.

In [ ]:
# FILTER: the three biggest misses on the test days
air_results = air.loc[Xa_test.index, ["city", "date", "pm25"]].copy()
air_results["predicted"] = air_predictions.round(1)
air_results["miss"] = (air_results["predicted"] - air_results["pm25"]).round(1)

air_results.reindex(air_results["miss"].abs().sort_values(ascending=False).index).head(3)

**Check against slide 12:** the worst miss is Houston on 2023-12-10: actual pm25 of 20.0, predicted
12.8, a miss of 7.2. The model only ever saw ozone, temperature, and humidity for that day.

In [ ]:
# COMMUNICATE: predicted vs. actual on the test days
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(air_results["pm25"], air_results["predicted"], color="#861F41")
both = pd.concat([air_results["pm25"], air_results["predicted"]])
ax.plot([both.min(), both.max()], [both.min(), both.max()], linestyle="--", color="gray")
ax.set_xlabel("Actual pm25")
ax.set_ylabel("Predicted pm25")
ax.set_title("Air quality test days: predicted vs. actual")
plt.tight_layout()
plt.show()

---
## Your Turn 5: A Baseline for the Lecture's Model

The lecture's model scored an R-squared of 0.330 and an MAE of 2.38. Build the same kind of baseline
used in Part 1 for the air-quality data: predict the average training-day `pm25` for every test day.
Then compare its MAE with the model's.

In [ ]:
# Your air-quality baseline here:

<details><summary>Show the answer</summary>

```python
average_pm25 = ya_train.mean()
air_baseline = [average_pm25] * len(ya_test)

print(f"Baseline R-squared: {r2_score(ya_test, air_baseline):.3f}   MAE: {mean_absolute_error(ya_test, air_baseline):.2f}")
print(f"Model    R-squared: {r2_score(ya_test, air_predictions):.3f}   MAE: {mean_absolute_error(ya_test, air_predictions):.2f}")
```

The baseline's MAE is about **2.99** micrograms per cubic meter, against the model's **2.38**, and its
R-squared is about 0. The model improves on guessing, but by much less than in Part 1: it cuts the
typical miss by about a fifth, not two-thirds. That matches the R-squared of 0.330. Ozone,
temperature, and humidity carry some information about pm25, and most of what drives it is
something the model was never given.

</details>

### Check Your Understanding

Answer each question in your head (or in a cell) before you open the worked answer.

**1. Reading the score.** Is an R-squared of 0.330 closer to 0 or to 1? What does that say about how
well ozone, temperature, and humidity explain pm25 on their own?

<details><summary>Show a worked answer</summary>

Closer to 0 than to 1. The three inputs explain roughly a third of the day-to-day variation in
pm25. That is real signal, but most of what drives air quality is something the model was never
given.
</details>

**2. Reading a miss.** Name one real-world factor about Houston on 2023-12-10 that the model never
saw but that could explain the miss.

<details><summary>Show a worked answer</summary>

Anything outside the three inputs: a traffic spike, an industrial or refinery event, wildfire smoke
drifting in, or a weather inversion trapping pollution near the ground. A model's mistakes often
point at what it was never given.
</details>

**3. Learned vs. written.** How is `model.fit()` different from a function you wrote by hand
earlier this semester?

<details><summary>Show a worked answer</summary>

A hand-written function encodes a rule you chose (a condition, a formula, a grouping).
`model.fit()` is given examples with known answers and works out the rule, here the weights,
that best matches them.
</details>

---
## Common Errors and What They Mean

| Error message (last line) | Usual cause | Fix |
|---------------------------|-------------|-----|
| `ModuleNotFoundError: No module named 'sklearn'` | scikit-learn is not installed in this environment | Run `%pip install scikit-learn` in a cell, then restart the kernel |
| `ValueError: Expected a 2-dimensional container but got <class 'pandas.Series'>` | Inputs given with single brackets, `df["col"]` | Use double brackets for inputs: `df[["col"]]` |
| `NotFittedError: This LinearRegression instance is not fitted yet` | `predict()` called before `fit()`, often after a kernel restart | Run the `fit()` cell first, or rerun the notebook from the top |
| `ValueError: The feature names should match those that were passed during fit` | The DataFrame given to `predict()` has a misspelled column, a missing column, or columns in a different order | Use exactly the same column names, in the same order, as in `fit()` |
| `KeyError: 'temp_range'` (or another name) | A column name is misspelled or never created | Check the spelling against `df.columns` |
| `FileNotFoundError: data.csv` | The notebook and `data.csv` are in different folders | Put `data.csv` in the same folder as this notebook |

When an error appears, read the **last line** of the message first. It names the problem; the lines
above it show where Python was when the problem happened.

---
## What Changed, What Didn't

| Operation | All semester | This week |
|-----------|--------------|-----------|
| Represent | Load data into a DataFrame | Same |
| Write the rule | We wrote the condition or formula | `model.fit()` learned it from examples |
| Apply it | Run our function on every row | `model.predict()` applies the learned rule |
| Aggregate | `groupby` + mean | R-squared and MAE summarize the misses |
| Communicate | We chose the plot | Same |

---
## Where This Goes Next

This was a preview. ADS 5525 / CS 5046 is where you choose between models, predict categories,
and evaluate more carefully. For now: finish your final project and get ready for Seminar 5.